# Fine-tuning con LoRA de Pixtral 12B — Entrada: imagen + texto

## Importar librerías

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import json
import os
import gc
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_curve,
    auc
)

from transformers import (
    LlavaForConditionalGeneration,
    AutoProcessor,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset as TorchDataset

from pyevall.evaluation import PyEvALLEvaluation
from pyevall.metrics.metricfactory import MetricFactory


## Configuración y parámetros

In [2]:
os.environ["HF_TOKEN"] = ""

MODEL_NAME = "mistral-community/pixtral-12b"
MAIN_PATH  = ".."
GROUP_ID   = "BeingChillingWeWillWin"
MODEL_ID   = "pixtral_12b_ft"

TEXT_COLUMN  = "combined_text"
LABEL_COLUMN = "label"

DATA_TRAIN_PATH = os.path.join(MAIN_PATH, "preprocessed_data", "train_split.json")
DATA_VAL_PATH   = os.path.join(MAIN_PATH, "preprocessed_data", "dev_split.json")
DATA_TEST_PATH  = os.path.join(MAIN_PATH, "preprocessed_data", "test_split.json")

DATA_BASE_DIR   = os.path.join(MAIN_PATH, "materials", "dataset_task2_exist2026")
OUTPUT_DIR      = os.path.join(MAIN_PATH, "weights", f"Pixtral-12B_{TEXT_COLUMN}_lora")
SAVE_PATH       = os.path.join(MAIN_PATH, "weights", f"Pixtral-12B_{TEXT_COLUMN}_final")
PREDICTIONS_DIR = os.path.join(MAIN_PATH, "predictions")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

MAX_LENGTH     = 8192
MAX_NEW_TOKENS = 10
NUM_EPOCHS     = 3
LEARNING_RATE  = 2e-5
BATCH_SIZE     = 1
GRAD_ACCUM     = 16
MAX_IMAGE_SIZE = 512

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

label_map         = {"NO": 0, "YES": 1}
label_map_inverse = {0: "NO", 1: "YES"}


## Carga y preprocesamiento de datos

In [3]:
def load_json_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return pd.DataFrame(data.values())

def build_combined_text(row):
    img_desc = str(row.get('image_description', '') or '')
    txt      = str(row.get('text', '') or '')
    return f"descripcion imagen: {img_desc}. Texto: {txt}"

train_df = load_json_dataset(DATA_TRAIN_PATH)
val_df   = load_json_dataset(DATA_VAL_PATH)
test_df  = load_json_dataset(DATA_TEST_PATH)

for df in [train_df, val_df, test_df]:
    df[TEXT_COLUMN] = df.apply(build_combined_text, axis=1)

train_df["label_int"] = train_df[LABEL_COLUMN].map(label_map)
val_df["label_int"]   = val_df[LABEL_COLUMN].map(label_map)
test_df["label_int"]  = -1

print(f"Text column used : {TEXT_COLUMN}")
print(f"Ejemplo de entrada:\n  {train_df[TEXT_COLUMN].iloc[0][:200]}")
print(f"\nTrain size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")
print(f"\nDistribución de etiquetas en TRAIN:")
print(train_df[LABEL_COLUMN].value_counts())
print(f"\nDistribución de etiquetas en VAL:")
print(val_df[LABEL_COLUMN].value_counts())


Text column used : combined_text
Ejemplo de entrada:
  descripcion imagen: a close up of a snake with its mouth open and its tongue out. Texto: Demostración de que las cosas mas peligrosas del mundo tienen el mismo aspecto. mémenoides 

Train size: 2146 | Val size: 537 | Test size: 687

Distribución de etiquetas en TRAIN:
label
YES    1282
NO      864
Name: count, dtype: int64

Distribución de etiquetas en VAL:
label
YES    321
NO     216
Name: count, dtype: int64


## Carga del modelo con cuantización y configuración LoRA

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           
    bnb_4bit_compute_dtype=DTYPE,       
    bnb_4bit_use_double_quant=True,  
)

print(f"Cargando modelo {MODEL_NAME}...")

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Cargando modelo mistral-community/pixtral-12b...


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

The image processor of type `PixtralImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


OutOfMemoryError: CUDA out of memory. Tried to allocate 80.00 MiB. GPU 0 has a total capacity of 44.40 GiB of which 63.00 MiB is free. Process 2497531 has 6.07 GiB memory in use. Including non-PyTorch memory, this process has 38.26 GiB memory in use. Of the allocated memory 37.42 GiB is allocated by PyTorch, and 361.53 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Dataset y Data Collator para Fine-tuning

In [ ]:
def resize_image_if_needed(image, max_size=MAX_IMAGE_SIZE):
    width, height = image.size
    if max(width, height) > max_size:
        if width > height:
            new_width, new_height = max_size, int(height * max_size / width)
        else:
            new_height, new_width = max_size, int(width * max_size / height)
        image = image.resize((new_width, new_height), Image.Resampling.LANCZOS)
    return image


def build_messages(text):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": f"Analyze this meme. {text}\n\nIs this meme sexist? Answer YES or NO."}
            ]
        }
    ]


class SexismMemeDataset(TorchDataset):
    def __init__(self, df, processor, base_dir, max_length=8192, max_image_size=512):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.base_dir = Path(base_dir)
        self.max_length = max_length
        self.max_image_size = max_image_size

    def __len__(self):
        return len(self.df)

    def _load_image(self, img_path):
        try:
            return resize_image_if_needed(Image.open(img_path).convert('RGB'), self.max_image_size)
        except Exception as e:
            print(f"[WARN] Error cargando imagen {img_path}: {e}")
            return Image.new('RGB', (224, 224), color='gray')

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = self._load_image(self.base_dir / row['path_memes'])
        target_text = "YES" if row['label_int'] == 1 else "NO"

        messages = build_messages(row[TEXT_COLUMN])
        prompt = self.processor.apply_chat_template(messages, add_generation_prompt=True)

        # Encode prompt-only (with image) to find the exact boundary for label masking
        prompt_enc = self.processor(text=prompt, images=image, return_tensors="pt", padding=False)
        prompt_length = prompt_enc['input_ids'].shape[1]

        # Encode full sequence (prompt + answer)
        encoding = self.processor(text=prompt + target_text, images=image, return_tensors="pt", padding=False)

        item = {}
        for k, v in encoding.items():
            if k in ['pixel_values', 'image_sizes']:
                item[k] = v
            elif isinstance(v, torch.Tensor):
                item[k] = v.squeeze(0)
            else:
                item[k] = v

        if item['input_ids'].shape[0] > self.max_length:
            item['input_ids'] = item['input_ids'][:self.max_length]
            item['attention_mask'] = item['attention_mask'][:self.max_length]

        labels = item['input_ids'].clone()
        labels[:prompt_length] = -100
        item['labels'] = labels
        return item


In [ ]:
class DataCollatorForVLM:
    def __init__(self, processor):
        self.pad_id = processor.tokenizer.pad_token_id

    @staticmethod
    def _pad(tensors, pad_value):
        max_len = max(t.shape[0] for t in tensors)
        return torch.stack([
            torch.cat([t, torch.full((max_len - t.shape[0],), pad_value, dtype=t.dtype)])
            for t in tensors
        ])

    def __call__(self, features):
        batch = {}
        if 'input_ids' in features[0]:
            batch['input_ids'] = self._pad([f['input_ids'] for f in features], self.pad_id)
        if 'attention_mask' in features[0]:
            batch['attention_mask'] = self._pad([f['attention_mask'] for f in features], 0)
        if 'labels' in features[0]:
            batch['labels'] = self._pad([f['labels'] for f in features], -100)
        if 'pixel_values' in features[0]:
            batch['pixel_values'] = torch.cat([f['pixel_values'] for f in features], dim=0)
        if 'image_sizes' in features[0]:
            sizes = [f['image_sizes'] for f in features]
            batch['image_sizes'] = torch.cat(sizes, dim=0) if isinstance(sizes[0], torch.Tensor) else sizes
        return batch


In [ ]:
train_dataset = SexismMemeDataset(train_df, processor, DATA_BASE_DIR, MAX_LENGTH, MAX_IMAGE_SIZE)
eval_dataset  = SexismMemeDataset(val_df,   processor, DATA_BASE_DIR, MAX_LENGTH, MAX_IMAGE_SIZE)

print(f"Train: {len(train_dataset)} | Val: {len(eval_dataset)}")

sample = train_dataset[0]
print(f"input_ids shape : {sample['input_ids'].shape}")
if 'pixel_values' in sample:
    print(f"pixel_values    : {sample['pixel_values'].shape}")
print(f"tokens con loss : {(sample['labels'] != -100).sum().item()}")


## Configuración del Trainer

In [ ]:
data_collator = DataCollatorForVLM(processor)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_total_limit=2,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✓ Trainer configurado")


## Entrenamiento

In [ ]:
print("Iniciando entrenamiento...")
trainer.train()

print("\nGuardando modelo...")
model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)
print(f"✓ Modelo guardado en: {SAVE_PATH}")


## Inferencia en DEV y cálculo de métricas

In [ ]:
@torch.no_grad()
def generate_prediction(row, model, processor, base_dir, max_image_size=MAX_IMAGE_SIZE):
    try:
        image = resize_image_if_needed(
            Image.open(Path(base_dir) / row['path_memes']).convert('RGB'), max_image_size
        )
    except:
        return {'classification': 'NO', 'confidence': 0.0}

    messages = build_messages(row[TEXT_COLUMN])
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)
    prompt_length = inputs['input_ids'].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=processor.tokenizer.pad_token_id
    )
    response = processor.tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True).strip().upper()
    return {'classification': 'YES' if 'YES' in response else 'NO', 'confidence': 0.9}


def save_probs_json(ids, probs, split_name, labels=None):
    records = [{'id': str(i), 'prob_YES': round(float(p), 6)} for i, p in zip(ids, probs)]
    if labels is not None:
        for rec, lbl in zip(records, labels):
            rec['label'] = label_map_inverse[int(lbl)]
    path = os.path.join(PREDICTIONS_DIR, f'{GROUP_ID}_{MODEL_ID}_probs_{split_name}.json')
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def evaluate_in_chunks(df, model, processor, base_dir, chunk_size=100, desc="Inferencia"):
    all_results = []
    n_chunks = (len(df) + chunk_size - 1) // chunk_size
    for chunk_idx in range(n_chunks):
        chunk_df = df.iloc[chunk_idx * chunk_size : (chunk_idx + 1) * chunk_size]
        chunk_results = []
        for _, row in tqdm(chunk_df.iterrows(), total=len(chunk_df), desc=f"{desc} [{chunk_idx+1}/{n_chunks}]"):
            try:
                pred = generate_prediction(row, model, processor, base_dir)
                chunk_results.append({
                    'id_EXIST': str(row['id_EXIST']),
                    'classification': pred['classification'],
                    'confidence': pred['confidence'],
                    'true_label': label_map_inverse.get(row.get('label_int', -1), 'UNKNOWN')
                })
            except Exception as e:
                print(f"[ERROR] id {row.get('id_EXIST', '?')}: {e}")
                chunk_results.append({
                    'id_EXIST': str(row['id_EXIST']),
                    'classification': 'NO',
                    'confidence': 0.0,
                    'true_label': label_map_inverse.get(row.get('label_int', -1), 'UNKNOWN'),
                    'error': str(e)
                })
        all_results.extend(chunk_results)
        checkpoint_path = os.path.join(PREDICTIONS_DIR, f'checkpoint_{desc.lower().replace(" ", "_")}_chunk{chunk_idx+1}.json')
        with open(checkpoint_path, 'w') as f:
            json.dump(all_results, f, indent=2)
    return all_results


In [ ]:
model.eval()
val_results    = evaluate_in_chunks(val_df, model, processor, DATA_BASE_DIR, chunk_size=100, desc="DEV")
val_results_ok = [r for r in val_results if 'error' not in r]

y_pred  = [label_map[r['classification']] for r in val_results_ok]
y_true  = [label_map[r['true_label']]     for r in val_results_ok]
y_probs = [r['confidence'] if r['classification'] == 'YES' else 1 - r['confidence'] for r in val_results_ok]

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)

save_probs_json(
    [r['id_EXIST'] for r in val_results_ok],
    y_probs, 'dev',
    labels=[label_map[r['true_label']] for r in val_results_ok]
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

fpr, tpr, _ = roc_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)
print(f"AUC      : {roc_auc:.4f}")

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0, 1]); plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve DEV — Pixtral 12B Fine-tuned')
plt.legend(loc="lower right"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Evaluación en DEV con PyEvALL

In [ ]:
dev_preds_for_pyevall = [
    {'test_case': 'EXIST2025', 'id': str(r['id_EXIST']), 'value': r['classification']}
    for r in val_results_ok
]
dev_preds_df = pd.DataFrame(dev_preds_for_pyevall)
dev_preds_path = os.path.join(PREDICTIONS_DIR, 'dev_predictions_temp.json')
with open(dev_preds_path, 'w', encoding='utf-8') as f:
    f.write(dev_preds_df.to_json(orient='records'))

valid_ids_set = set(r['id_EXIST'] for r in val_results_ok)
dev_gold = [
    {'test_case': 'EXIST2025', 'id': str(id_exist), 'value': label}
    for id_exist, label in zip(val_df['id_EXIST'].values, val_df[LABEL_COLUMN].values)
    if str(id_exist) in valid_ids_set
]
dev_gold_df = pd.DataFrame(dev_gold)
dev_gold_path = os.path.join(PREDICTIONS_DIR, 'dev_gold_temp.json')
with open(dev_gold_path, 'w', encoding='utf-8') as f:
    f.write(dev_gold_df.to_json(orient='records'))

test_eval = PyEvALLEvaluation()
metrics = [MetricFactory.Accuracy.value, MetricFactory.FMeasure.value]
report = test_eval.evaluate(dev_preds_path, dev_gold_path, metrics)
print("\n=== Evaluación en DEV con PyEvALL ===")
report.print_report()


## Inferencia en TEST y generación de predicciones finales

In [ ]:
print("Evaluando en TEST (en fragmentos)...")

test_results = evaluate_in_chunks(test_df, model, processor, DATA_BASE_DIR, chunk_size=100, desc="TEST")

test_results_ok = [r for r in test_results if 'error' not in r]
print(f"\nResultados válidos: {len(test_results_ok)}/{len(test_results)}")

y_probs_test = [r['confidence'] if r['classification'] == 'YES' else (1 - r['confidence']) for r in test_results_ok]
test_preds = [r['classification'] for r in test_results_ok]

valid_test_ids = [r['id_EXIST'] for r in test_results_ok]
save_probs_json(valid_test_ids, y_probs_test, 'test')

print(f"\nPredicciones en TEST:")
print(f"  Total: {len(test_preds)}")
print(f"  YES  : {sum(1 for p in test_preds if p == 'YES')} ({100*sum(1 for p in test_preds if p == 'YES')/len(test_preds):.2f}%)")
print(f"  NO   : {sum(1 for p in test_preds if p == 'NO')} ({100*sum(1 for p in test_preds if p == 'NO')/len(test_preds):.2f}%)")


## Guardar predicciones en formato PyEvALL para TEST

In [ ]:
test_preds_for_submission = [
    {'test_case': 'EXIST2025', 'id': str(r['id_EXIST']), 'value': r['classification']}
    for r in test_results_ok
]
test_preds_df = pd.DataFrame(test_preds_for_submission)

output_filename = f"{GROUP_ID}_{MODEL_ID}.json"
output_path = os.path.join(PREDICTIONS_DIR, output_filename)

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(test_preds_df.to_json(orient='records'))

print(f"\n✓ Predicciones guardadas en: {output_path}")
print(f"  Total muestras: {len(test_preds_for_submission)}")

if len(test_results) > len(test_results_ok):
    errors_path = os.path.join(PREDICTIONS_DIR, f"{GROUP_ID}_{MODEL_ID}_errors.json")
    test_errors = [r for r in test_results if 'error' in r]
    with open(errors_path, 'w') as f:
        json.dump(test_errors, f, indent=2)
    print(f"⚠ {len(test_errors)} errores guardados en: {errors_path}")
